# Data Reduction
In this activity, you will be reducing the data we took last week. In this notebook, you will learn how to go from raw data taken on a telescope and create science ready images! 

By the end of this activity, you will be ready to create a pretty 3-color Red-Blue-Green (RGB) image like this one:

![image](rsw-0023-RGB.jpg)

Credit: Richard S. Wright Jr.

**A note on how this notebook is organized:** all of the file paths and frame numbers you need live in **one configuration cell** near the top (Section 0). You edit that cell once, and every other cell below reads from it automatically. You should never need to type a file path by hand anywhere else in this notebook -- if you find yourself doing that, look for a helper function that can do it for you instead.

## Step 0: Load the tools we need
Before we can do anything, we need to load some tools that other people already wrote for us. Think of it like grabbing tools out of a toolbox before you start a project. Run the cell below once, and you're set.

- `reduction`: our own toolbox for cleaning up pictures from the Nickel 1-m telescope
- `astropy.io.fits`: lets us open and save telescope picture files (called FITS files)
- `matplotlib.pyplot`: lets us draw the pictures so we can actually see them
- `numpy`: helps with math and combining pictures together
- `shutil`: helps us copy files from one folder to another
- `glob`: helps us grab a whole list of files at once
- `os`: lets us check that a file really exists before we try to use it

In [ ]:
# load our tools

import os
import reduction
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
import shutil
import glob

## Step 0.1: Fill in your info (just once!)
This is the only spot in the whole notebook where you'll type in file locations and frame numbers by hand. Grab your observing log from telescope night and fill in the blanks below.

Why does this matter? If everyone types file names by hand in a dozen different cells, it's really easy to make a typo, accidentally overwrite someone else's file, or end up with file names that don't match your partner's. By filling this in just once, every cell after this will always use the exact same names -- so there's only one place to check if something looks off.

In [ ]:
# ============================================================
# YOUR INFO GOES HERE -- fill this in once, using your observing log
# ============================================================

# Where are your raw (original) files? Don't forget the / at the end!
source_dir =   #<<<EDIT

# These are the standard folders (or "directories") for this dataset -- you shouldn't need to change these
calibration_dir = source_dir + '../calibration/'
science_dir = source_dir + '../science/'

# The four filters (colors) we took pictures through
FILTERS = ['b', 'v', 'r', 'halpha']
FILTER_DIRS = {filt: calibration_dir + filt + '_flat/' for filt in FILTERS}
bias_dir = calibration_dir + 'bias/'
flats_dir = calibration_dir + 'flats/'

# Look at your observing log and fill in the frame numbers below.
# range(3, 10) means frames 3 through 9 -- so if your bias frames
# are numbers 3 through 9, you'd write bias_frames = range(3, 10)
bias_frames =    #<<<EDIT THESE
b_flat_frames = 
v_flat_frames = 
r_flat_frames = 
halpha_flat_frames = 
science_frames =    # every science frame, any object, any filter

FLAT_FRAMES = {
    'b': b_flat_frames,
    'v': v_flat_frames,
    'r': r_flat_frames,
    'halpha': halpha_flat_frames,
}

# Which object did you pick, and which frame number is it in each filter?
# If you didn't take a picture of your object in one of the filters, leave it as None.
object_name = 'YOUR OBJECT NAME HERE'   # for example: 'M51', 'M57', or 'NGC5634'
science_frame_numbers = {
    'b': ,
    'v': ,
    'r': ,
    'halpha': None,
}

## Step 0.2: Two tools we already built for you
Two things happen over and over in this notebook: finding the right file, and drawing a picture of it so we can look at it. Instead of making you write that same code again and again, we already built two little tools (called *functions*) that do it for you:

- `get_file_from_frame_num(...)` finds the right file for you, just from a frame number. It also double-checks the file is really there -- so if you typed a frame number wrong, you get a clear message right away instead of a confusing error much later.
- `show_image(...)` draws a picture of the data, with a colorbar and labels, so it's easy to look at.

You don't need to understand every single line inside these -- just know what they do, and use them like a tool. Run the cell below once.

In [ ]:
def get_file_from_frame_num(frame_num, suffix='', directory=None):
    """
    Finds the file that matches a given frame number.

    frame_num : the frame number, like 12
    suffix    : which cleaning step it's from --
                ''       the original raw file
                '_os'     after removing the overscan
                '_bs'     after removing the bias
                '_ff'     after flat-fielding
                '_bp'     after removing bad pixels
    directory : which directory to look in (uses source_dir if you don't say)
    """
    directory = directory if directory is not None else source_dir
    path = f'{directory}d{frame_num}{suffix}.fits'
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Can't find {path}. Double check your frame number, "
            f"the suffix, and that you've actually run this step yet."
        )
    return path


def show_image(data, vmin=0, vmax=1000, title=None):
    """Draws a picture of a FITS image so we can look at it."""
    plt.imshow(data, origin='lower', vmin=vmin, vmax=vmax)
    plt.colorbar(label='Counts')
    plt.xlabel('X [pixels]')
    plt.ylabel('Y [pixels]')
    if title:
        plt.title(title)
    plt.show()

Let's try to use get_file_from_frame_num() to make sure it works. Type in a frame number from your observing log (e.g. `bias_frames[0]` for the first bias frame or `b_flat_frames[1]` for the second B filter flat frame) and run the cell. You should see the full path to that file printed out. Check to make sure it's the right file.

In [ ]:
# Check the file path

# Select a frame number to look at. 
# You can change this to any frame number you want to inspect (e.g. bias_frames[1], b_flat_frames[0], etc.).
frame_num =    #<<<EDIT
print(f"Looking for frame number: {frame_num}")

# Now we use the get_file_from_frame_num() function to find the path to that file. 
# If it exists, it will print the full path; if not, it will raise an error.
file_path = get_file_from_frame_num(frame_num)
print(file_path)

## Step 1: Overscan subtraction
Every image our camera takes has a few extra columns of pixels along the edge that were never actually pointed at the sky. Since no light hits them, we can use them to measure the camera's natural "starting point" -- kind of like weighing an empty bowl before you weigh the food inside it. This starting point is called the **bias level**, and it can drift a little throughout the night.

We use those extra edge columns to correct for that drift. This step is called an **overscan subtraction**. Once we're done with it, we don't need those extra columns anymore, so they get trimmed off.

### Let's look at a raw picture first
Pick any raw file and let's take a look. You should be able to spot two overscan regions: a bright stripe of "bad" pixels, and a fainter stripe next to it. You might need to play with the brightness scaling below to see them clearly.

In [ ]:
frame_num = bias_frames[0]  #<<<EDIT. Try changing this to look at a different file (e.g. bias_frames[1], b_flat_frames[0], etc.)
file_path = get_file_from_frame_num(frame_num)    # get the path to that file using the function `get_file_from_frame_num()``
print(f"File path: {file_path}")

# Now we can load the data from that file using a handy function in the astropy library called `fits.getdata()`
data = fits.getdata(file_path)

# Here is some code to display the image. You can change the vmin and vmax values to make it easier to see the features in the image.
plt.imshow(data, origin='lower', vmin=1000, vmax=1200)  #<<<EDIT. Try changing vmax to make it easier to see
plt.colorbar(label='Counts')
plt.xlabel('X [pixels]')
plt.ylabel('Y [pixels]')
plt.show()

### That was a lot of lines just to look at one picture...
...and we're going to want to look at pictures a lot more in this notebook. That's exactly what `show_image()` is for! Watch -- this next cell makes the exact same plot as the one above, in way less code.

In [ ]:
data = fits.getdata(file_path)
show_image(data, vmin=1000, vmax=1200, title=f'Raw frame {frame_num}')

### Now let's actually remove the overscan
We'll use a tool from our toolbox called `overscan_subtraction()`. You just hand it a list of files, and it does the work for you. Files that come out of this step get `_os` added to their name (short for "overscan subtracted").

This might take a minute to run. While you wait, peek in your raw data directory -- can you see new files with the `_os` suffix showing up?

In [ ]:
files = source_dir + '*.fits'  # grabs every file in this directory that ends with .fits

reduction.overscan_subtraction(files)

### Now check your work
Let's look at the same frame again, but this time the overscan-subtracted version. The overscan stripes on the right should be gone (the bright bad-pixel stripe will still be there for now -- we'll deal with that later).

**Your turn:** use `get_file_from_frame_num()` and `show_image()` (both from Step 0.2) to plot the `_os` version of `frame_num`. Hint: `get_file_from_frame_num()` has a `suffix` option for exactly this.

In [ ]:
# use get_file_from_frame_num() and show_image() to plot the overscan-subtracted version of frame_num
# hint: get_file_from_frame_num(frame_num, suffix='_os')





## Step 2: Sort files into directories
Now that the overscan is removed, we need to sort our files into the right directories:
- `calibration` -- has a directory for `bias`, `b_flat`, `v_flat`, `r_flat`, and `halpha_flat`. This is where our calibration files go (the ones we use to measure and correct for camera quirks).
- `science` -- this is where our actual pictures of cool space objects go.

If we didn't know how to code, we would now have to drag and drop every single file manually. But because you already typed in all your frame numbers back in Step 0.1, this part can be completely automatic with the help of Python libraries and for loops! 

The cell below sorts every file into the correct directory for you. If a frame number turns out to be wrong, you'll get a friendly warning telling you exactly which one, instead of everything breaking.

We've included print statements so you can monitor the progress. If you want to double-check, you can also peek in the directories themselves to make sure the files are really there. See if you can follow the logic of the loops below.

In [ ]:
# We are going to define a dictionary called FRAME_GROUPS. 
# It maps each group of frames to the directory where they should go. 
# The keys are the group names, and the values are tuples of (frame_numbers, destination_directory). 
# We'll start with bias and science frames.
FRAME_GROUPS = {'bias': (bias_frames, bias_dir), 'science': (science_frames, science_dir)}

# Now we'll add the flat frames for each filter using a for loop.
for filt in FILTERS:
    print(f"Adding {filt} flat frames to FRAME_GROUPS")
    FRAME_GROUPS[filt] = (FLAT_FRAMES[filt], FILTER_DIRS[filt])

# With all this set up, we can now loop through each group and copy the files to the right directory.
for group_name, (frame_numbers, dest_dir) in FRAME_GROUPS.items():
    print(f"Copying {group_name} frames to {dest_dir}. Frames: {list(frame_numbers)}")
    for num in frame_numbers:
        src = get_file_from_frame_num(num, suffix='_os')
        try:
            shutil.copy2(src, dest_dir)
        except FileNotFoundError:
            print(f"Heads up: couldn't find frame {num} for '{group_name}' -- double check this frame number.")

## Step 3: Make a master bias
Remember, the **bias** is a picture taken with zero exposure time -- it tells us how each pixel behaves even when no light hits it at all. Every pixel is a tiny bit different, so instead of using just one bias picture, we combine several bias frames together. Combining them using the **median** (basically, the "middle" value at each pixel) smooths out noise and gives us one clean **master bias** to use for everything else.

In [ ]:
bias_files = glob.glob(bias_dir + '*_os.fits')    # grabs every file in this directory that ends with _os.fits
data_stack = []

# grab the data from every bias frame
for frame in bias_files:
    data_stack.append(fits.getdata(frame))

# combine them using the median (the middle value at each pixel)
medianBias = np.median(data_stack, axis=0)

# save the master bias, with a note in the header about what we did
header = fits.getheader(bias_files[0])
header['HISTORY'] = 'Median combined'
fits.writeto(bias_dir + 'bias.fits', medianBias, header, overwrite=True)

### Take a look at your master bias
What's the average count level? Is that a big number or a small number, compared to the most a pixel can hold (about 65,000)?


In [ ]:
# Calculate the average count level in the master bias frame. Look up the numpy function `np.mean()` to do this.
medianBias_average = EDIT_ME
fraction_of_65000 = EDIT_ME
print(f"The average count level in the master bias frame is: {medianBias_average}")
print(f"This is {fraction_of_65000:.2%} of the full well depth of 65,000 counts.")

Use `show_image()` to plot `medianBias`. It's already sitting in memory from the cell above, so you don't need to open the file again.

In [ ]:
# use show_image() to plot medianBias





## Step 4: Subtract the bias from every picture
The bias shows up in every single picture we take -- not just the bias frames, but also our flats and our science pictures. So before we can do anything else with those, we need to subtract the master bias from each one. Files that have had this done get `_bs` added to their name (bias-subtracted).

Let's walk through it together for the B flats first, so you can see exactly what's happening.

### Watch what happens with the B flats

In [ ]:
# grab every overscan-subtracted B flat
datafilesin = glob.glob(FILTER_DIRS['b'] + '*_os.fits')
print(f"Found {len(datafilesin)} B flat files to bias-subtract.")
print(f"Files: {datafilesin}")

# _bs means "bias subtracted" -- these will be the new file names
datafilesout = [i[:-5] + '_bs.fits' for i in datafilesin]
print(f"Output files will be named: {datafilesout}")

n = len(datafilesin)

# subtract the master bias from each flat, one at a time
for i in range(0, n):
    data, header = fits.getdata(datafilesin[i], header=True)
    dataout = data - medianBias
    header['HISTORY'] = 'Bias subtracted'
    fits.writeto(datafilesout[i], dataout, header, overwrite=True)

### Now let's do the same thing for everything else
The V, R, and Halpha flats -- and all of your science pictures -- need the exact same treatment. Since it's exactly the same steps, just repeated, the cell below does it automatically for all of them. Take a peek at the code -- can you spot how it's doing the same thing as the cell above, just for more folders?

In [ ]:
def bias_subtract_directory(directory):
    """Subtracts medianBias from every *_os.fits file in a directory."""
    datafilesin = glob.glob(directory + '*_os.fits')
    datafilesout = [i[:-5] + '_bs.fits' for i in datafilesin]
    for i in range(len(datafilesin)):
        data, header = fits.getdata(datafilesin[i], header=True)
        dataout = data - medianBias
        header['HISTORY'] = 'Bias subtracted'
        fits.writeto(datafilesout[i], dataout, header, overwrite=True)


for filt in FILTERS:
    print(f"Bias-subtracting {filt} flat frames in {FILTER_DIRS[filt]}")
    if filt == 'b':
        continue  # already done above
    bias_subtract_directory(FILTER_DIRS[filt])

bias_subtract_directory(science_dir)

## Step 5: Make a normalized flat-field for each filter
Even under perfectly even light, not every pixel on the camera responds exactly the same way -- some pixels are a little more or less sensitive, and this changes depending on which filter (color) we're using. A **flat-field** measures this pixel-by-pixel unevenness so we can divide it back out.

We call it "normalized" because we scale all the values so they average out to 1. That way, dividing by the flat corrects the unevenness without changing the overall brightness of our picture.

### First, let's see why this matters
Pick any science frame and look at its bias-subtracted version. Do you notice anything weird -- donut shapes, dark spots, uneven brightness across the picture?

**Your turn:** pick any frame number from `science_frame_numbers` and use `get_file_from_frame_num()` and `show_image()` to plot its `_bs` version.

In [ ]:
# pick a science frame number and use get_file_from_frame_num() and show_image() to plot its _os_bs version





### Now let's build the flats
We'll walk through the B flat together first.

In [ ]:
b_flist = glob.glob(FILTER_DIRS['b'] + '*_bs.fits')
bflat_stack = []

# read in each flat and normalize it by its own median
for file in b_flist:
    data, header = fits.getdata(file, header=True)
    data = data / np.median(data)
    bflat_stack.append(data)

# combine all the flats together, then normalize the final result too
bflat = np.median(bflat_stack, axis=0)
m = np.mean(bflat)
bflat = bflat / m
header['HISTORY'] = 'Combined and normalized flat field'
fits.writeto(flats_dir + 'bflat.fits', bflat, header, overwrite=True)

### And now the rest, automatically
Same steps, just repeated for V, R, and Halpha:

In [ ]:
FLATS = {'b': bflat}  # already made above

def make_master_flat(directory, filter_name):
    """Builds a normalized master flat from every *_bs.fits file in a folder."""
    flist = glob.glob(directory + '*_bs.fits')
    stack = []
    for file in flist:
        data, header = fits.getdata(file, header=True)
        data = data / np.median(data)
        stack.append(data)
    flat = np.median(stack, axis=0)
    flat = flat / np.mean(flat)
    header['HISTORY'] = 'Combined and normalized flat field'
    fits.writeto(flats_dir + filter_name + 'flat.fits', flat, header, overwrite=True)
    return flat


for filt in FILTERS:
    if filt == 'b':
        continue  # already done above
    FLATS[filt] = make_master_flat(FILTER_DIRS[filt], filt)

### Take a look at one of your flats
Can you see the pixel-by-pixel pattern?

**Your turn:** use `show_image()` to plot one of the flats in `FLATS`, like `FLATS['r']`.

In [ ]:
# use show_image() to plot one of the flats, e.g. FLATS['r']





## Step 6: Flat-field your science pictures
Now we can use our flats to fix the unevenness in your actual pictures of space! Remember the object and frame numbers you picked back in Step 0.1 -- everything below uses those automatically. Choose from:
- M51 (Whirlpool Galaxy)
- M57 (Ring Nebula)
- NGC5634 (a globular cluster -- a big ball of thousands of old stars)

First, let's look at your science pictures before we flat-field them. Not every object has pictures in all four filters, and that's okay.

**Your turn:** loop through `FILTERS`, and for each one that isn't `None` in `science_frame_numbers`, use `get_file_from_frame_num()` and `show_image()` to plot it. The loop is already set up below -- you just need to fill in the one line that does the plotting.

In [ ]:
for filt in FILTERS:
    frame_num = science_frame_numbers[filt]
    if frame_num is None:
        continue  # you didn't take a picture in this filter
    # use get_file_from_frame_num() (with suffix='_os_bs') and show_image() to plot this frame




### Now let's actually flat-field them
Same pattern as before -- watch it happen for B first.

In [ ]:
frame_num = science_frame_numbers['b']

if frame_num is not None:
    filein = get_file_from_frame_num(frame_num, suffix='_os_bs', directory=science_dir)
    fileout = filein[:-5] + '_ff.fits'

    data, header = fits.getdata(filein, header=True)
    dataout = data / FLATS['b']
    header['HISTORY'] = 'Flat Fielded'
    fits.writeto(fileout, dataout, header, overwrite=True)
else:
    print('No B-filter picture for this object -- nothing to do here.')

### And the rest, automatically

In [ ]:
def flat_field_frame(frame_num, filt):
    """Divides one science picture by the matching master flat."""
    filein = get_file_from_frame_num(frame_num, suffix='_os_bs', directory=science_dir)
    fileout = filein[:-5] + '_ff.fits'
    data, header = fits.getdata(filein, header=True)
    dataout = data / FLATS[filt]
    header['HISTORY'] = 'Flat Fielded'
    fits.writeto(fileout, dataout, header, overwrite=True)


for filt in FILTERS:
    if filt == 'b':
        continue  # already done above
    frame_num = science_frame_numbers[filt]
    if frame_num is None:
        continue
    flat_field_frame(frame_num, filt)

### Compare before and after
What's different about your pictures now that they're flat-fielded?

**Your turn:** loop through `FILTERS` again and plot the `_ff` version of each one.

In [ ]:
for filt in FILTERS:
    frame_num = science_frame_numbers[filt]
    if frame_num is None:
        continue
    # use get_file_from_frame_num() (with suffix='_os_bs_ff') and show_image() to plot this frame





## Step 7: Remove the bad pixels
Remember that bright stripe of "bad" pixels we saw way back in Step 1? Time to get rid of it! We'll use a tool from our toolbox called `remove_bad_pixels()`, which already knows exactly which columns on this camera are bad and fixes them for you. Files from this step get `_bp` added to their name.

In [ ]:
files_in = glob.glob(science_dir + '*_ff.fits')
files_out = [file[:-5] + '_bp.fits' for file in files_in]

reduction.remove_bad_pixels(files_in, files_out)

### Take a look at your final, cleaned pictures
What do you notice, compared to before?

**Your turn:** loop through `FILTERS` one more time and plot the `_bp` version of each one -- this is your final, cleaned data!

In [ ]:
for filt in FILTERS:
    frame_num = science_frame_numbers[filt]
    if frame_num is None:
        continue
    # use get_file_from_frame_num() (suffix='_os_bs_ff_bp') and show_image() to plot this frame





# Step 8: Make a color picture
You've done all the hard work -- now for the fun part! We'll use a free tool called JS9 to combine your red, green, and blue pictures into one color image, just like the cameras on big space telescopes do.

Open JS9 here: https://js9.si.edu

### Load your pictures
Click the File tab, then "open local." Find your cleaned pictures (the ones ending in `_os_bs_ff_bp.fits`) and upload them. They'll show up in the Images list.

![image](./js9/step2.png)

### Turn on RGB mode
Click the Color tab, scroll down, and click "rgb mode."

![image](./js9/step1.png)

### Assign red, green, and blue
Click on one of your images in the File tab, then go to the Color tab. Pick Red, Green, or Blue depending on which filter that picture was taken through.

![image](./js9/step3.png)

### You're all set!
Play around with the contrast on each color by right-clicking and dragging on the image. When you're happy with it, you should end up with a picture that looks something like this!

![image](./js9/rgb.png)

# Bonus activity: Matplotlib Playground

Now that you've done all the hard work of cleaning up your pictures, you can play around with how they look and annotate them! 

So far, we've been using `show_image()` to display our pictures, hiding all the lines of code that actually make the picture. In the following cells, you'll get to see how to use `matplotlib.pyplot` directly to display your images. You can search online for more information about `matplotlib` and how to use it to customize your plots. 

Note that we named `matplotlib.pyplot` as `plt` when we imported it, so you'll need to use `plt` instead of `matplotlib.pyplot` in the code below.

In [ ]:
frame_num = science_frame_numbers['b']
file_path = get_file_from_frame_num(frame_num, suffix='_os_bs_ff_bp', directory=science_dir)
data = fits.getdata(file_path)

# You've already seen these few lines in the previous cells!
plt.imshow(data, origin='lower', vmin=1000, vmax=1200)
img = plt.colorbar(label='Counts')

# Try to change the title, xlabel, and ylabel to something more descriptive
plt.xlabel('')
plt.ylabel('')
plt.title('', fontsize=80)   # you might want to change the fontsize too!

# You can also change the color map to something else. Try 'magma', 'plasma', or 'inferno' for some nice options. 
# How about 'jet' or 'cubehelix'? 
cbar = plt.colorbar(img, cmap='magma')

# BONUS bonus activity: Can you add a red star marker at the center of the image?
# Hint: use plt.plot(x, y, marker='*', color='red')
# How would you find the center of the image for x and y? Look up how to use data.shape to find the exact center pixel location!


plt.show()